# Plán letu nad údolím Vltavy: GSD, překryvy, terén a vítr

[![Otevřít v Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/flight_planner_cs.ipynb)

**OpenSkyLab · notebook k laboratoři** [Plánovač letu dronu](https://openskylab.space/edu_viz_flight_planner.html) · [English version](flight_planner_en.ipynb)

V laboratoři jsi plánoval let nad rovinou a nad modelovým kopcem. Tady stejné vzorce použiješ na skutečný terén: svah údolí Vltavy u Roztok z digitálního modelu reliéfu ČÚZK DMR 5G (lidar, buňka 2 m) a na aktuální předpověď větru ve výšce letu.

**Cíl:** spočítat výšku letu pro cílové GSD, rozestup linek, interval a rychlost, rozmazání a zkosení, ukázat, jak terén mění výšku nad zemí a překryv, a naplánovat návrat proti větru.

**Data:** DMR 5G (ČÚZK, služba ImageServer, výšky v Bpv, buňka 2 m) pro území 1,2 × 0,8 km na levém svahu údolí Vltavy u Roztok; hodinová předpověď větru ve 10, 80 a 120 m z Open-Meteo; parametry kamer z technických listů výrobců (stejné jako v laboratoři).

**Úkoly:** pět. Odpovědi piš do buněk *Tvoje odpověď* pod úkoly. Řádky označené `# ◀ parametr` můžeš měnit a buňku spustit znovu (Shift + Enter).

**Čas:** asi 50 min. Potřebuješ Google účet (Colab); nic se neinstaluje do počítače. Buňky spouštěj popořadě shora dolů.

## 1. Příprava

**Povinná buňka, spusť ji jako první.** Doinstaluje knihovnu rasterio, která čte rastrová data (GeoTIFF) přímo z internetu, a načte knihovny. Definuje `cuzk_raster`, která stáhne výřez rastru ČÚZK (model terénu DMR 5G nebo povrchu DMP 1G) jako pole výšek. Definuje `open_meteo`, která se ptá služby Open-Meteo na počasí a při chybě nebo přetížení to zkusí znovu s rostoucí pauzou. Nic zatím nestahuje. Kód je sbalený; zobrazíš ho pomocí **Show code**.

In [ ]:
# @title Příprava (spusť jako první) {display-mode: "form"}
%pip install -q rasterio

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from pyproj import Transformer


def cuzk_raster(service, x0, y0, x1, y1, res=2.0):
    """Výřez rastru ČÚZK (dmr5g, dmp1g, dmp_obrazova_korelace) v S-JTSK (EPSG:5514), výšky v Bpv.
    Vrací pole výšek (řádek 0 = sever) a jeho transformaci."""
    nx, ny = int(round((x1 - x0) / res)), int(round((y1 - y0) / res))
    url = f"https://ags.cuzk.cz/arcgis2/rest/services/{service}/ImageServer/exportImage"
    params = {
        "bbox": f"{x0},{y0},{x1},{y1}",
        "bboxSR": 5514,
        "imageSR": 5514,
        "size": f"{nx},{ny}",
        "format": "tiff",
        "pixelType": "F32",
        "interpolation": "RSP_BilinearInterpolation",
        "f": "image",
    }
    for attempt in range(4):
        r = requests.get(url, params=params, timeout=120)
        if r.ok and r.headers.get("content-type", "").startswith("image"):
            break
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    with rasterio.MemoryFile(r.content) as mem, mem.open() as ds:
        return ds.read(1).astype(float), ds.transform


def open_meteo(params, tries=6):
    """Dotaz na Open-Meteo; při chybě nebo vyčerpaném limitu to zkusí znovu."""
    for i in range(tries):
        try:
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast", params=params, timeout=60
            )
            if r.ok:
                return r.json()
            reason = r.json().get("reason", r.status_code)
        except requests.RequestException as e:
            reason = type(e).__name__
        print("Open-Meteo:", reason, "– zkusím znovu")
        time.sleep(5 * (i + 1))
    raise RuntimeError("Open-Meteo neodpovídá, spusť buňku za chvíli znovu.")

**Povinná buňka, spusť ji hned po přípravě.** Vložení nastavení je nepovinné: když do pole `Z_VIZUALU` vložíš nastavení zkopírované z laboratoře, převezmou buňky níže stejné hodnoty parametrů a u každé to vypíšou. Když pole necháš prázdné (`{}`), použijí se výchozí hodnoty. Buňku spusť v obou případech, protože definuje funkci `z_viz`, kterou používají další buňky. Nic nestahuje. Kód je sbalený; zobrazíš ho pomocí **Show code**.

In [ ]:
# @title Nastavení z laboratoře (spusť; vložení je nepovinné) {display-mode: "form"}
# @markdown V laboratoři klikni na odkaz Notebook. Nastavení se zkopíruje; vlož ho sem místo `{}` (Ctrl+V) a buňku spusť.
Z_VIZUALU = {}  # @param {type:"raw"}
STEM = "flight_planner"


def z_viz(name, default, mapping=None):
    """Hodnota parametru z laboratoře, pokud ji vložený kód obsahuje; jinak výchozí hodnota."""
    z = (
        Z_VIZUALU
        if isinstance(Z_VIZUALU, dict) and Z_VIZUALU.get("osl") == STEM
        else {}
    )
    if name not in z:
        return default
    v = z[name]
    if mapping is not None:
        if v not in mapping:
            print(
                f"{name}: hodnota {v!r} z laboratoře v tomto notebooku není, ponechávám {default!r}"
            )
            return default
        v = mapping[v]
    elif isinstance(default, bool):
        v = bool(v)
    elif isinstance(default, int):
        v = int(round(v))
    elif isinstance(default, float):
        v = float(v)
    print(f"{name} = {v!r}  (z laboratoře)")
    return v


if isinstance(Z_VIZUALU, dict) and Z_VIZUALU and Z_VIZUALU.get("osl") != STEM:
    print(
        f"Vložený kód je z jiné laboratoře ({Z_VIZUALU.get('osl')}), notebook ho nepoužije."
    )

> **💻 Co dělá kód:** Uloží parametry čtyř kamer: rozměry čipu, počet pixelů, ohniskovou vzdálenost a nejkratší interval mezi snímky.

In [ ]:
# kamery: šířka a výška čipu (mm), počet pixelů, ohnisková vzdálenost (mm), nejkratší interval mezi snímky (s)
CAMERAS = {
    "P1 35 mm": dict(sw=35.9, sh=24.0, pw=8192, ph=5460, f=35.0, tmin=0.7),
    "Mavic 3E": dict(sw=17.3, sh=13.0, pw=5280, ph=3956, f=12.3, tmin=0.7),
    "Mavic 3M (MS)": dict(sw=5.2, sh=3.9, pw=2592, ph=1944, f=4.34, tmin=2.0),
    "RGB 1″ 20 Mpx": dict(sw=13.2, sh=8.8, pw=5472, ph=3648, f=8.8, tmin=2.0),
}

## 2. Kamera, výška a GSD

GSD = H · p / f, kde p je velikost pixelu na čipu (šířka čipu / počet pixelů). Z cílového GSD tedy H = GSD · f / p.

> **💻 Co dělá kód:** Pro cílové GSD `gsd_target` spočítá u každé kamery velikost pixelu na čipu, potřebnou výšku letu a velikost záběru na zemi. Ukáže tabulku.

In [ ]:
gsd_target = 1.5  # ◀ parametr: cílové GSD v cm/px
rows = []
for name, c in CAMERAS.items():
    p = c["sw"] / c["pw"]  # mm
    H = gsd_target / 100 * c["f"] / p  # m
    rows.append(
        {
            "kamera": name,
            "pixel na čipu (µm)": p * 1000,
            "výška letu (m)": H,
            "záběr napříč (m)": c["pw"] * gsd_target / 100,
            "záběr podél (m)": c["ph"] * gsd_target / 100,
        }
    )
cam_tab = pd.DataFrame(rows).set_index("kamera")
cam_tab.round(1)

> **✏️ Úkol 1.** Proč musí Mavic 3M pro stejné GSD letět mnohem níž než P1? Nastav `gsd_target` na 2 a pak na 4 cm/px. Pro které kamery a GSD by výška překročila 120 m, a tedy limit otevřené kategorie? Jaké má nízký let důsledky (počet snímků, čas, bezpečnost)?

*Tvoje odpověď:* 

## 3. Skutečný terén a plán linek

Načteme DMR 5G pro svah údolí Vltavy. Linky vedou ve směru východ–západ, tedy po spádnici dolů a zpět nahoru. Místo vzletu je na planině na západním okraji.

> **💻 Co dělá kód:** Stáhne z ČÚZK model terénu DMR 5G v buňce 2 m pro území 1200 × 800 m, určí výšku místa vzletu na západním okraji a vykreslí terén s vrstevnicemi po 10 m.

In [ ]:
cx, cy = Transformer.from_crs(4326, 5514, always_xy=True).transform(14.392, 50.152)
W, Hh = 1200, 800  # území: šířka (V–Z) a výška (S–J) v m
dem, tr = cuzk_raster("dmr5g", cx - W / 2, cy - Hh / 2, cx + W / 2, cy + Hh / 2, res=2)
z_takeoff = float(np.median(dem[:, :10]))  # horní, západní okraj území
print(
    f"terén {dem.min():.1f}–{dem.max():.1f} m n. m. (Bpv), místo vzletu {z_takeoff:.1f} m n. m."
)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(dem, cmap="terrain", extent=(0, W, 0, Hh))
fig.colorbar(im, label="nadmořská výška (m, Bpv)")
ax.contour(
    np.flipud(dem),
    levels=np.arange(np.floor(dem.min() / 10) * 10, dem.max() + 10, 10),
    colors="k",
    linewidths=0.4,
    extent=(0, W, 0, Hh),
)
ax.set(
    xlabel="m (západ → východ)",
    ylabel="m",
    title="DMR 5G, levý svah údolí Vltavy u Roztok",
)
plt.show()

> **💻 Co dělá kód:** Pro zvolenou kameru, výšku, překryvy a rychlost (mohou přijít z laboratoře) spočítá GSD, vzdálenost mezi snímky (bázi), rozestup linek, počet linek a snímků, interval mezi snímky a čas letu. Upozorní, pokud kamera interval nestihne.

In [ ]:
# Kamera.
cam = CAMERAS[
    z_viz(
        "cam",
        "Mavic 3E",
        {
            "rgb": "RGB 1″ 20 Mpx",
            "p1": "P1 35 mm",
            "m3e": "Mavic 3E",
            "m3m": "Mavic 3M (MS)",
        },
    )
]  # ◀ parametr
H0 = z_viz("H0", 80)  # ◀ parametr: výška letu nad místem vzletu (m)
fo, so = z_viz("fo", 0.80), z_viz("so", 0.70)  # ◀ parametr: podélný a příčný překryv
v_air = z_viz("v_air", 10.0)  # ◀ parametr: rychlost vůči vzduchu (m/s)
long_across = True  # ◀ parametr: delší strana snímku napříč letem?

p = cam["sw"] / cam["pw"] / 1000  # m
n_across, n_along = (cam["pw"], cam["ph"]) if long_across else (cam["ph"], cam["pw"])
gsd0 = H0 * p / (cam["f"] / 1000)  # GSD nad místem vzletu
base = (1 - fo) * n_along * gsd0  # vzdálenost mezi snímky
spacing = (1 - so) * n_across * gsd0  # rozestup linek
n_lines = int(np.ceil(Hh / spacing)) + 1
interval = base / v_air
photos = n_lines * (int(W / base) + 1)
t_flight = (n_lines * W + (n_lines - 1) * spacing) / v_air
print(
    f"GSD nad místem vzletu {gsd0 * 100:.2f} cm/px, báze {base:.1f} m, rozestup linek {spacing:.1f} m"
)
print(
    f"linek {n_lines}, snímků {photos}, interval {interval:.2f} s (kamera zvládne {cam['tmin']} s), čas letu {t_flight / 60:.1f} min bez otoček"
)
if interval < cam["tmin"]:
    print(f"POZOR: kamera nestihne; nejvyšší rychlost je {base / cam['tmin']:.1f} m/s")

> **✏️ Úkol 2.** Zvyšuj `v_air`, dokud se neobjeví upozornění, že kamera nestihne. Jaká je nejvyšší rychlost? Přepni `long_across = False`: jak se změní báze, nejvyšší rychlost, počet linek a čas letu? Proč otočený snímek dovolí rychlejší let, a přesto může let trvat déle? Nastav podélný překryv 0,85 jako pro vegetaci.

*Tvoje odpověď:* 

## 4. Stálá výška, nebo sledování terénu?

Při stálé výšce nad místem vzletu se výška nad zemí mění s terénem: H_AGL = H0 + z_vzlet − z. S ní se mění GSD i záběr, ale báze a rozestup linek zůstávají, takže se mění skutečný překryv.

> **💻 Co dělá kód:** Pro každou buňku terénu spočítá výšku nad zemí, GSD a skutečný podélný a příčný překryv, buď při stálé výšce nad místem vzletu, nebo při sledování terénu (`follow`). Vypíše jejich rozsah a podíl území výš než 120 m nad terénem a vykreslí tři mapy.

In [ ]:
follow = False  # ◀ parametr: True = sledování terénu (stálá výška nad zemí H0)
agl = np.full_like(dem, H0) if follow else H0 + z_takeoff - dem
gsd = agl * p / (cam["f"] / 1000)
fo_local = 1 - base / (n_along * gsd)
so_local = 1 - spacing / (n_across * gsd)
print(
    f"výška nad zemí {agl.min():.0f}–{agl.max():.0f} m, GSD {gsd.min() * 100:.2f}–{gsd.max() * 100:.2f} cm/px"
)
print(
    f"podélný překryv {fo_local.min():.0%}–{fo_local.max():.0%}, příčný {so_local.min():.0%}–{so_local.max():.0%}"
)
print(f"plocha nad 120 m nad terénem: {np.mean(agl > 120):.0%} území")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, (arr, lbl, cm) in zip(
    ax,
    [
        (agl, "výška nad zemí (m)", "viridis"),
        (gsd * 100, "GSD (cm/px)", "magma"),
        (so_local * 100, "příčný překryv (%)", "RdYlGn"),
    ],
):
    im = a.imshow(arr, cmap=cm, extent=(0, W, 0, Hh))
    fig.colorbar(im, ax=a, label=lbl)
    a.set_xticks([])
    a.set_yticks([])
ax[0].set_title(
    "sledování terénu" if follow else f"stálá výška {H0} m nad místem vzletu"
)
plt.show()

> **✏️ Úkol 3.** Při stálé výšce 80 m nad místem vzletu: jaký je rozsah GSD a překryvů? Kde je překryv nejmenší, nahoře u místa vzletu, nebo v nejnižších místech, a proč? Kolik území je výš než 120 m nad terénem? Přepni `follow = True`: co se zlepší a co k sledování terénu potřebuješ (model terénu, jeho přesnost, překážky)?

*Tvoje odpověď:* 

## 5. Skutečný vítr, rozmazání a postupná závěrka

Z Open-Meteo vezmeme předpověď větru pro toto území na dnešek. Rozmazání = rychlost vůči zemi · expozice / GSD; zkosení postupnou závěrkou = rychlost vůči zemi · doba vyčtení čipu / GSD. Zenmuse P1 a Mavic 3E mají mechanickou závěrku, takže u nich zkosení prakticky odpadá (t_read = 0); výchozích 30 ms je řádová hodnota pro elektronickou postupnou závěrku.

> **💻 Co dělá kód:** Stáhne z Open-Meteo dnešní hodinovou předpověď větru v 10, 80 a 120 m, směru větru v 80 m a nárazů pro území a vypíše hodiny od 6 do 19.

In [ ]:
w = open_meteo(
    {
        "latitude": 50.152,
        "longitude": 14.392,
        "forecast_days": 1,
        "wind_speed_unit": "ms",
        "timezone": "Europe/Prague",
        "hourly": "wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_direction_80m,wind_gusts_10m",
    }
)
met = pd.DataFrame(w["hourly"]).set_index("time")
print(met.iloc[6:20].to_string())

> **💻 Co dělá kód:** Pro zvolenou hodinu rozloží vítr v 80 m na složku podél linek a napříč nimi a spočítá rychlost vůči zemi na lince na východ a na západ. Z ní a z expozice `t_exp` a doby vyčtení čipu `t_read` (mohou přijít z laboratoře) vypíše rozmazání a zkosení v pixelech.

In [ ]:
# Hodina letu (místní čas); výchozí je největrnější hodina mezi 6 a 20 h.
hour = 6 + int(met.wind_speed_80m.values[6:21].argmax())  # ◀ parametr
t_exp = z_viz("t_exp", 1 / 1000)  # ◀ parametr: expoziční čas (s)
# Doba vyčtení čipu u postupné závěrky (s); 0 = globální závěrka.
t_read = z_viz("t_read", 0.030)  # ◀ parametr
wind = met.wind_speed_80m.iloc[hour]
wdir = met.wind_direction_80m.iloc[hour]
# linky vedou V–Z; složka větru podél linek a napříč nimi
along = wind * np.cos(np.radians(wdir - 90))
cross = wind * np.sin(np.radians(wdir - 90))
for leg, sign in [("linka na východ", -1), ("linka na západ", +1)]:
    # Rychlost vůči zemi (vítr odkud → znaménko).
    gs = np.sqrt(max(v_air**2 - cross**2, 0)) + sign * along
    print(
        f"{leg}: vůči zemi {gs:4.1f} m/s, rozmazání {gs * t_exp / gsd0:.2f} px, zkosení {gs * t_read / gsd0:.1f} px"
    )
print(
    f"{met.index[hour]}: vítr v 80 m: {wind:.1f} m/s od {wdir:.0f}°; složka k východu {-along:+.1f} m/s, k severu {cross:+.1f} m/s"
)

> **✏️ Úkol 4.** Jaký vítr hlásí předpověď ve 10, 80 a 120 m a proč se liší? Jak se liší rozmazání a zkosení na lince po větru a proti větru? Proč zkosení nezmenší kratší expozice? Kdy bys linky raději otočil podél větru, nebo napříč?

*Tvoje odpověď:* 

## 6. Baterie a návrat domů

Mavic 3E má baterii 77 Wh a výrobce udává až 45 min letu bez větru, tedy průměrně asi 100 W. Spočítáme, kolik energie potřebuje návrat z nejvzdálenějšího bodu, když vítr fouká od místa vzletu.

> **💻 Co dělá kód:** Odhadne energii na misi z průměrného výkonu dronu a na návrat z nejvzdálenějšího rohu území proti větru. Porovná jejich součet s využitelnou energií baterie po započtení teploty a rezervy a vypíše, zda baterie stačí.

In [ ]:
E_bat = 77.0  # ◀ parametr: energie baterie (Wh)
P = 77.0 / (45 / 60)  # průměrný výkon (W) z udávané výdrže
# 1,0 při 20 °C; v mrazu zhruba 0,7–0,8.
temp_factor = 1.0  # ◀ parametr: využitelná kapacita
reserve = 0.20  # ◀ parametr: rezerva při přistání
headwind = wind  # nejhorší případ: celý vítr proti návratu
d_home = np.hypot(W, Hh)  # nejvzdálenější roh území (m)
gs_home = v_air - headwind
E_mission = P * t_flight / 3600
E_return = P * d_home / max(gs_home, 0.1) / 3600
E_use = E_bat * temp_factor * (1 - reserve)
print(
    f"mise {E_mission:.1f} Wh + návrat {E_return:.1f} Wh (proti větru {headwind:.1f} m/s, vůči zemi {gs_home:.1f} m/s) = {E_mission + E_return:.1f} Wh"
)
print(
    f"využitelná energie {E_use:.1f} Wh → {'stačí' if E_mission + E_return <= E_use else 'NESTAČÍ, rozděl misi'}"
)

> **✏️ Úkol 5.** Stačí jedna baterie? Nastav `temp_factor = 0.75` a pak zvyš vítr (`headwind = 8`). Proč je „tam po větru, zpátky proti větru“ past? Zkus také vyšší `v_air`: proč při silném větru pomáhá letět rychleji, i když výkon roste?

*Tvoje odpověď:* 

> **✏️ Rozšíření (nepovinné).** Spolužáci mohou mít notebook pro jiné území (Roztoky, Brno-Bystrc, Hosín u Hluboké). Porovnejte převýšení, rozsah výšky nad zemí při stálé výšce 80 m nad místem vzletu a podíl území nad 120 m. Kde by stálá výška byla nejméně vhodná a kolik baterií by na území potřeboval každý z vás?

*Tvoje odpověď:* 

---
### Zdroje dat a literatura

- Digitální model reliéfu České republiky 5. generace (DMR 5G), © ČÚZK, licence CC BY 4.0, služba https://ags.cuzk.cz/arcgis2/rest/services/dmr5g/ImageServer
- Počasí: Open-Meteo.com (CC BY 4.0), https://open-meteo.com
- Parametry kamer: technické listy DJI Zenmuse P1, Mavic 3 Enterprise (3E, 3M) a obecná kamera 1″ 20 Mpx (u Mavic 3M a obecné kamery je nejkratší interval 2 s orientační hodnota); baterie Mavic 3 Enterprise 5000 mAh, 15,4 V (77 Wh).
- Výška letu v otevřené kategorii max. 120 m od nejbližšího bodu zemského povrchu: prováděcí nařízení Komise (EU) 2019/947, příloha, UAS.OPEN.010.

### Jak citovat

OpenSkyLab (2026). Plán letu nad údolím Vltavy: GSD, překryvy, terén a vítr [notebook, výukový materiál], verze 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/flight_planner_cs.ipynb. Licence CC BY 4.0.

Text a kód notebooku: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data mají licence svých poskytovatelů (viz výše).

*Materiál připravil tým OpenSkyLab s pomocí generativní AI Claude (Anthropic), která pomáhala s návrhem kódu a textu. Za obsah odpovídají autoři.*